# 6.5 My own vectors: does Louis say different things in French than in English?

> **Question:** before Benji was born, did Louis talk about different things in French
> than in English?

**My expectation:** in French he shares more love and compliments; his English is more
businesslike and practical.

Only the weeks **before Benji's birth** (`settings.BIRTH_DATE`): after that, Louis writes
almost only in French (68% French, 7% English), so there is hardly any English left to
compare with.

The method, from lessons 6.1–6.4:

1. **One row** = one *unit*: the messages of one session (6.4's time gap), split by
   language, so a session with three English and two French messages gives two units.
2. **Vectors** from a *multilingual* sentence encoder (6.3, 6.4), so a French and an
   English unit that mean the same thing get the same vector.
3. **k-means** on those vectors: groups of units that are about the same thing.
4. **Per week**, compare how French and English units spread over those groups, and check
   the difference with a shuffle that holds the week and the unit length fixed (04.6).

> Needs `uv sync --extra huggingface` once. The first run downloads the encoder (~470 MB).

In [ ]:
import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")  # joblib's core count crashes on Windows

import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import Pipeline
from goad_toolkit.visualizer import LinePlot, PlotSettings, ProjectionPlot, ScreePlot
from notebooktester import param
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score

from roos_pipeline import settings
from roos_pipeline.settings import Paths
from roos_pipeline.vectors import (
    LanguageDumbbellPlot,
    StripQuotedReply,
    WeeklyLanguagePanels,
    WeekStart,
    WordCount,
    cluster_contrast,
    cluster_examples,
    language_units,
    weekly_cluster_shares,
)
from scripts.sessionize import fit_session_threshold

LANGUAGE_NAMES = {"fr": "French", "en": "English"}

## 6.5.1 Louis's messages before Benji

From `2_features.parquet` (the pipeline's stage 2), which already carries `language`
(`py3langid`, en/nl/fr, under 3 characters = "unknown"). `StripQuotedReply` first: a reply
that quotes one of my messages would otherwise put my words in Louis's mouth.

In [ ]:
features = pd.read_parquet(Paths().features)
before_benji = features[(features["author"] == "Louis") & (features["ts_local"] < settings.BIRTH_DATE)]
louis = Pipeline().add(StripQuotedReply).add(WeekStart).apply(before_benji.copy())
louis = louis[louis["message"] != ""]

print(f"{len(louis):,} messages from Louis, {louis['week'].nunique()} weeks, "
      f"{louis['ts_local'].min():%d %b %Y} to {louis['ts_local'].max():%d %b %Y}")
louis["language"].value_counts()

## 6.5.2 What is one row? Sessions, split by language

6.4's lesson: one message ("Top", "Hahaha") is too short to mean anything as a vector. So
messages are merged into sessions first, with the gap threshold *fitted* on Louis's own
gaps (`fit_session_threshold`, the lesson 4 exponential), not guessed.

Sessions are cut on all his messages, Dutch included, so switching language mid-conversation
does not start a new session. Only then is each session split into its French and English
part. Dutch is left out of the comparison.

In [ ]:
threshold = fit_session_threshold(louis["ts_local"], louis["author"])
print(f"session threshold: {threshold:.0f} s ({threshold / 60:.1f} minutes)")

units = Pipeline().add(WordCount).apply(language_units(louis, threshold))
units.groupby("language")[["n_messages", "n_words"]].median()

**The first thing to know:** French units are much shorter than English ones. That is not a
detail. A short text lands in a different part of the vector space than a long one, *whatever
it says*: in a first try, k-means with 2 groups split the units into short and long, not by
meaning. Two things handle that:

- units under `settings.MIN_UNIT_WORDS` words are left out (below);
- the shuffle test in 6.5.6 compares French and English only *within the same length band*.

What the 4-word minimum throws away, most common first:

In [ ]:
short = units[units["n_words"] < settings.MIN_UNIT_WORDS]
for language, group in short.groupby("language"):
    top = group["message"].str.lower().str.strip().value_counts().head(12)
    print(f"{LANGUAGE_NAMES[language]}: {len(group):,} short units dropped, e.g. {top.to_dict()}")

units = units[units["n_words"] >= settings.MIN_UNIT_WORDS].reset_index(drop=True)
units.groupby(["language", "length_band"]).size().unstack()

## 6.5.3 Vectors: why a multilingual encoder

6.4 used `all-MiniLM-L6-v2`. That model was trained on English only. For this question that is
fatal: it would put every French unit in one corner because it is *French*, and k-means would
find the language, not what was said. `settings.EMBED_MODEL` was trained so that a sentence and
its translation get the same vector. Four sentences show the difference:

In [ ]:
pairs = ["je t'aime mon amour", "I love you my love", "à quelle heure part le train ?", "what time does the train leave?"]
for name in ["sentence-transformers/all-MiniLM-L6-v2", settings.EMBED_MODEL]:
    v = SentenceTransformer(name).encode(pairs, normalize_embeddings=True)
    print(f"{name.split('/')[-1]:>40s}:  love fr~en {v[0] @ v[1]:.2f}   train fr~en {v[2] @ v[3]:.2f}   "
          f"love~train (fr) {v[0] @ v[2]:.2f}")

In [ ]:
UNIT_SAMPLE = param(None, test=300)  # notebooktester runs on a small sample
if UNIT_SAMPLE is not None:
    units = units.sample(n=min(UNIT_SAMPLE, len(units)), random_state=settings.SEED).reset_index(drop=True)

encoder = SentenceTransformer(settings.EMBED_MODEL)
vectors = encoder.encode(units["message"].tolist(), batch_size=64, normalize_embeddings=True,
                         show_progress_bar=False)
print(f"{vectors.shape[0]:,} units x {vectors.shape[1]} dimensions")

## 6.5.4 300 dimensions?

The number of dimensions is not a choice you make when embedding: the encoder decides it,
384 here. "300 dimensions" can only mean: *reduce* the 384 to 300 with PCA before k-means.
Whether that is a good idea, the scree plot (6.1) answers.

In [ ]:
pca = PCA().fit(vectors)
cumulative = np.cumsum(pca.explained_variance_ratio_)

scree = PlotSettings(figsize=(8, 4), title="How much of the 384 dimensions do the first n components hold?",
                     xlabel="number of components", ylabel="share of variance")
fig, ax = ScreePlot(scree).plot(explained_variance_ratio=pca.explained_variance_ratio_)
ax.axvline(300, color=settings.GREY, linestyle="--")
ax.set_xticks([1, 50, 100, 150, 200, 250, 300, 384])
ax.lines[0].set_markersize(2)  # 384 components: dots, not beads

for n in [2, 50, 100, 300]:
    if n <= len(cumulative):
        print(f"{n:>4d} components: {cumulative[n - 1]:.1%} of the variance")
print(f"90% needs {np.searchsorted(cumulative, 0.9) + 1} components")

In [ ]:
K_CHECK = 6
full = KMeans(K_CHECK, n_init=10, random_state=settings.SEED).fit_predict(vectors)
reduced = KMeans(K_CHECK, n_init=10, random_state=settings.SEED).fit_predict(
    PCA(n_components=min(300, len(vectors) - 1)).fit_transform(vectors)
)
print(f"k-means on 384 vs. on 300 PCA components, same k: adjusted Rand index "
      f"{adjusted_rand_score(full, reduced):.2f} (1 = the same groups)")

**What this means for "300 dimensions":** *(read the two numbers above first, then answer in
your own words: what does going from 384 to 300 buy, and what does it cost? Would another
number of components be a better reason to reduce, or none at all?)*

*Answer:* …

## 6.5.5 k-means: how many groups?

The vectors are normalised (length 1), so k-means' straight-line distance ranks pairs the same
way cosine does (6.3). Silhouette per k, on the vectors themselves, never on a 2D picture (6.1.6).

In [ ]:
K_RANGE = range(2, 11)
silhouettes = pd.DataFrame({
    "k": list(K_RANGE),
    "silhouette": [
        silhouette_score(vectors, KMeans(k, n_init=10, random_state=settings.SEED).fit_predict(vectors), metric="cosine")
        for k in K_RANGE
    ],
})
line = PlotSettings(figsize=(7, 3.5), title="Silhouette per k (1 = sharp groups, 0 = none)", xlabel="k",
                    ylabel="silhouette")
fig, ax = LinePlot(line).plot(data=silhouettes, x="k", y="silhouette", marker="o", color=settings.GREY)
ax.set_ylim(0, None)
silhouettes.round(3).T

The silhouette stays close to 0 for every k: Louis's units do **not** fall into sharp,
separate groups. k-means will still hand out groups, but they are cuts through one continuous
cloud, like dividing a colour gradient into bands. That is fine for this question (*how much of
the French vs. the English sits in each band*), as long as nobody reads the groups as "the six
kinds of thing Louis talks about".

`K` below is a starting point (6), not a decision: choose it yourself, after reading the
examples. Fewer groups are easier to name; more groups can split "love" from "compliments".

In [ ]:
K = 6  # my choice, because: ...
kmeans = KMeans(K, n_init=10, random_state=settings.SEED).fit(vectors)
units["cluster"] = kmeans.labels_

pd.crosstab(units["cluster"], units["language"].map(LANGUAGE_NAMES), margins=True)

### What is each group about?

The units closest to each group's centre, per language: the most typical members, not a
random draw. Read them and **name the groups yourself** in `CLUSTER_NAMES`. Check also whether
a group means the same thing in French and in English, or whether it mostly holds one language.

In [ ]:
for language in settings.COMPARE_LANGUAGES:
    mask = (units["language"] == language).to_numpy()
    examples = cluster_examples(units.loc[mask, "message"], vectors[mask], units.loc[mask, "cluster"].to_numpy(),
                                kmeans.cluster_centers_, n=6)
    print(f"===== {LANGUAGE_NAMES[language]} =====")
    for k, texts in examples.items():
        print(f"--- group {k}")
        for text in texts:
            print("   ", text)

In [ ]:
CLUSTER_NAMES: dict[int, str] = {
    # 0: "...",
}
names = {k: CLUSTER_NAMES.get(k, f"group {k}") for k in range(K)}

### The cloud, seen from above

A PCA picture of the 384 dimensions, only to *look*: the groups were made in all 384, and the
first two components hold a small share of the variance (see 6.5.4), so overlap here does not
mean overlap in the data.

In [ ]:
coords = PCA(n_components=2).fit_transform(vectors)
panel = PlotSettings(figsize=(12, 5.5), title=f"Louis's units before Benji, PCA ({cumulative[1]:.0%} of the variance)",
                     xlabel="", ylabel="", max_cols=2,
                     subplot_titles=["coloured by k-means group", "coloured by language"])
host = ProjectionPlot(panel)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(ProjectionPlot(panel), axes[0], coordinates=coords,
                  labels=units["cluster"].map(names).to_numpy(), palette="colorblind", s=6,
                  hue_order=[names[k] for k in range(K)])
_ = host.plot_on_axes(ProjectionPlot(panel), axes[1], coordinates=coords,
                  labels=units["language"].map(LANGUAGE_NAMES).to_numpy(),
                  palette={"French": settings.FRENCH_COLOR, "English": settings.ENGLISH_COLOR}, s=6)

## 6.5.6 French against English, all weeks together

For each group: the share of Louis's French units in it, and the share of his English units.
The p-value comes from shuffling the French/English label 10,000 times **within each week and
length band** (`cluster_contrast`): a week keeps its own mix, a length band keeps its own, so
what is left is the difference that week and length cannot explain (`beyond_strata`).

In [ ]:
N_SHUFFLE = param(settings.N_SHUFFLE, test=200)
contrast = cluster_contrast(units, n_shuffle=N_SHUFFLE)

dumbbell = PlotSettings(figsize=(9, 0.6 * K + 1.5),
                        title="Which groups does Louis use more in French, which more in English?",
                        xlabel="share of his units in that language (%)", ylabel="")
fig, ax = LanguageDumbbellPlot(dumbbell).plot(contrast=contrast, names=names,
                                              pair_labels={"fr": "French", "en": "English"})
n_lang = units["language"].map(LANGUAGE_NAMES).value_counts()
fig.text(0.01, -0.02, f"Units of {settings.MIN_UNIT_WORDS}+ words before Benji's birth: "
         f"{n_lang.get('French', 0):,} French, {n_lang.get('English', 0):,} English, "
         f"{units['week'].nunique()} weeks. p: shuffle within week x length band, {N_SHUFFLE:,}x.",
         fontsize=8, color="dimgray")

contrast.rename(index=names).style.format({
    "share_fr": "{:.1%}", "share_en": "{:.1%}", "difference": "{:+.1%}", "shuffle_mean": "{:+.1%}",
    "beyond_strata": "{:+.1%}", "p": "{:.4f}",
})

## 6.5.7 Week by week

The same comparison per week: one panel per group, the share of that week's French units in
it against the share of that week's English units. A week counts for a language only with at
least `settings.MIN_UNITS_PER_WEEK` units in that language; otherwise one unit makes it 0% or 100%.
Dots are the real weeks, the line a centred 4-week average. Dashed lines: the positive test
(Aug 2024) and the move (Nov 2024).

In [ ]:
weekly = weekly_cluster_shares(units)
weeks_per_language = weekly.groupby("language")["week"].nunique()
print(f"weeks with enough units: French {weeks_per_language.get('fr', 0)}, English {weeks_per_language.get('en', 0)}")

panels = PlotSettings(figsize=(13, 2.6 * ((K + 1) // 2)), title="Share of the week's units, French vs. English",
                      xlabel="", ylabel="%", max_cols=2, sharex=True,
                      subplot_titles=[names[k] for k in range(K)])
fig, axes = WeeklyLanguagePanels(panels).plot(n_plots=K, weekly=weekly, clusters=list(range(K)),
                                              pair_labels={"fr": "French", "en": "English"})
for ax in axes:
    for day in [settings.TEST_DATE, settings.MOVE_DATE]:
        ax.axvline(day, color=settings.LIGHT_GREY, linestyle="--", linewidth=1)

## What I found

*(in my own words, after naming the groups)*

1. **Love and compliments:** which group(s) are they? Is Louis's French share there higher than
   his English share, and does it survive the shuffle (`beyond_strata`, `p`)?
2. **Practical / businesslike:** which group(s)? Higher in English?
3. **Week by week:** is the difference there in most weeks, or does a handful of weeks carry it?
   Does it change after the test or the move (dashed lines)?
4. **Did my expectation hold?** What surprised me?

## Things to check before I believe it

- **Language labels.** `py3langid` calls "hahaha" French and "ouiiii" English. How often is a
  unit's language wrong, or mixed (a French unit with English words in it)? Read 20 at random.
- **Does the encoder still "see" the language?** The PCA picture coloured by language shows it.
  If a group is almost all one language, is that because of *what* was said or *that* it was French?
- **k.** Does the answer to 1 and 2 stay the same with `K` one lower and one higher?
- **The 4-word minimum.** Rerun with `settings.MIN_UNIT_WORDS = 2`: does "bonne nuit 😘" change
  the love result?
- **n = 1 person.** Whatever this shows, it is about Louis in this chat, not about French and English.